# Análise Completa de Caso e Investigação de Gargalos da Modelagem (Caso 2 - PMC3437073_01)

Este notebook aplica o pipeline de ETL e construção do Grafo de Conhecimento clínico (desenvolvido em `complete_case_analysis.ipynb`) sobre o **Caso 2 (`PMC3437073_01`)**.

O objetivo central desta análise é demonstrar empiricamente **gargalos, anomalias e limitações estruturais** da modelagem clássica baseada em regras e grafos estáticos quando confrontada com um relato de caso complexo que envolve:
* Negação de histórico de contato epidemiológico;
* Transições temporais de esquemas terapêuticos (fase intensiva vs. fase de manutenção);
* Lesões espaciais comunicantes (fístula transdiafragmática);
* Coexistência de testes laboratoriais negativos com confirmação histopatológica positiva;
* Distinção entre infecção remota cicatrizada (*"old pulmonary tuberculosis"*) e abscesso agudo atual.


### Carregamento do Caso 2

In [1]:
import pandas as pd
import diplo_grafo.config as cfg
import re
from diplo_grafo.sentencing import get_sentences
from diplo_grafo.sentence import Sentence

# Ajuste para a pasta do projeto
base = cfg.SAMPLE_DATA_DIR

# Carrega os CSVs
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")
data_dict = pd.read_csv(base / "data_dictionary.csv")

# Seleção do Caso 2
case = cases.iloc[2]

sentences = get_sentences(case)

print(f"ID do Caso: {case['case_id']} | Idade: {case['age']} | Gênero: {case['gender']}")
print("\n--- TEXTO DO CASO ---")
print(case['case_text'])
print("\n==================================\n")
print("\n|| ".join([s.text for s in sentences]) + " ||\n")


2026-09-08 23:52:00.705 | INFO     | diplo_grafo.config:<module>:11 - PROJ_ROOT path is: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1


ID do Caso: PMC3437073_01 | Idade: 55.0 | Gênero: Male

--- TEXTO DO CASO ---
A 55-year-old male presented with a gradually increasing mass on the left lateral lower chest and upper abdomen over a period of one year. He had no other medical problems and no history of contact with a tuberculous patient. On examination, a lobulated subcutaneous mass measuring about 7 cm in diameter, soft to cystic in consistency, immobile and attached to the skin which was tense and slightly red, tender but not warm, not pulsating, and with no bruit was found on the left lower chest and upper abdomen laterally. The spleen was enlarged 8 cm below the costal margin. Other systems were normal.
Chest X-rays showed a calcified lesion near the hilum of the right lung suggestive of an old pulmonary tuberculosis. Abdominal radiographs showed calcified lesions in the left upper quadrant (Figure 1). Ultrasound of the abdomen indicated an enlarged spleen with areas of calcification. CT scan of the abdomen revealed 

In [2]:
for s in sentences:
    s.tokenize()
    print(s.tokens)


['A', '55', 'year-old', 'male', 'presented', 'with', 'a', 'gradually', 'increasing', 'mass', 'on', 'the', 'left', 'lateral', 'lower', 'chest', 'and', 'upper', 'abdomen', 'over', 'a', 'period', 'of', 'one', 'year']
['He', 'had', 'no', 'other', 'medical', 'problems', 'and', 'no', 'history', 'of', 'contact', 'with', 'a', 'tuberculous', 'patient']
['On', 'examination', ',', 'a', 'lobulated', 'subcutaneous', 'mass', 'measuring', 'about', '7 cm', 'in', 'diameter', ',', 'soft', 'to', 'cystic', 'in', 'consistency', ',', 'immobile', 'and', 'attached', 'to', 'the', 'skin', 'which', 'was', 'tense', 'and', 'slightly', 'red', ',', 'tender', 'but', 'not', 'warm', ',', 'not', 'pulsating', ',', 'and', 'with', 'no', 'bruit', 'was', 'found', 'on', 'the', 'left', 'lower', 'chest', 'and', 'upper', 'abdomen', 'laterally']
['The', 'spleen', 'was', 'enlarged', '8 cm', 'below', 'the', 'costal', 'margin']
['Other', 'systems', 'were', 'normal']
['Chest', 'X-rays', 'showed', 'a', 'calcified', 'lesion', 'near', '

In [3]:
from diplo_grafo.clinical_terms import reconstruct_clinical_tokens

for sentence in sentences:
    sentence.tokenize()
    sentence.tokens = reconstruct_clinical_tokens(sentence.tokens)

for sentence in sentences:
    print(sentence.tokens)


['A', '55', 'year-old', 'male', 'presented', 'with', 'a', 'gradually', 'increasing', 'mass', 'on', 'the', 'left', 'lateral', 'lower', 'chest', 'and', 'upper', 'abdomen', 'over', 'a', 'period', 'of', 'one', 'year']
['He', 'had', 'no', 'other', 'medical', 'problems', 'and', 'no', 'history', 'of', 'contact', 'with', 'a', 'tuberculous', 'patient']
['On', 'examination', ',', 'a', 'lobulated', 'subcutaneous', 'mass', 'measuring', 'about', '7 cm', 'in', 'diameter', ',', 'soft', 'to', 'cystic', 'in', 'consistency', ',', 'immobile', 'and', 'attached', 'to', 'the', 'skin', 'which', 'was', 'tense', 'and', 'slightly', 'red', ',', 'tender', 'but', 'not', 'warm', ',', 'not', 'pulsating', ',', 'and', 'with', 'no', 'bruit', 'was', 'found', 'on', 'the', 'left', 'lower', 'chest', 'and', 'upper', 'abdomen', 'laterally']
['The', 'spleen', 'was', 'enlarged', '8 cm', 'below', 'the', 'costal', 'margin']
['Other', 'systems', 'were', 'normal']
['Chest', 'X-rays', 'showed', 'a', 'calcified', 'lesion', 'near', '

### Construção do Grafo de Conhecimento com Atributos Ricos (Nodes & Edges)

Aplicamos a mesma função de extração semântica com atributos ricos desenvolvida para o pipeline, utilizando os padrões de expressões regulares de `extracao_regex.ipynb` e o esquema do Mermaid em `docs/steps.md`:

In [4]:
# ==============================================================================
# LÓGICA DE CONSTRUÇÃO DO GRAFO CLÍNICO COM ATRIBUTOS RICOS (NODES & EDGES)
# ==============================================================================
import re
import pandas as pd
import diplo_grafo.config as cfg

# 1. Expressões Regulares de Extração de Atributos (derivadas de extracao_regex.ipynb)
DIMENSION_3D = re.compile(
    r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)",
    re.IGNORECASE
)
DIMENSION_2D = re.compile(
    r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)",
    re.IGNORECASE
)
DIMENSION_1D = re.compile(
    r"\b(\d+(?:\.\d+)?)\s*(cm|mm)\b",
    re.IGNORECASE
)
LAB_MEASUREMENT = re.compile(
    r"(\d+(?:,\d+)*(?:\.\d+)?)\s*(mg/dL|g/dL|ng/mL|U/L|iu/ml|mm/h|mmol/L|g%|fL|/uL|x10\^9/L|K/uL|%)\b",
    re.IGNORECASE
)
DURATION_HISTORY = re.compile(
    r"\b(\d+)-(day|week|month|year)\s+history\b",
    re.IGNORECASE
)
DURATION_GENERIC = re.compile(
    r"\b(\d+-(?:day|week|month|year))\b",
    re.IGNORECASE
)
DOSAGE = re.compile(
    r"\b(\d+(?:\.\d+)?)\s*(mg|mcg|g|IU|units)\b",
    re.IGNORECASE
)
TIMELINE_POD = re.compile(
    r"\b(?:POD|postoperative day)\s*(\d+)\b",
    re.IGNORECASE
)
LATERALITY = re.compile(
    r"\b(right|left|bilateral)\b",
    re.IGNORECASE
)
ASPECT = re.compile(
    r"\b(posterior|anterior|superior|inferior|distal|proximal)\b",
    re.IGNORECASE
)

# Padrão para detecção de escopo de negação ou resolução clínica
NEGATION_PATTERN = re.compile(
    r"(?:no\s+evidence\s+of|negative\s+for|free\s+of|denies|denied|without|resolution\s+of|resolved|uninvolved)\s+([^.,;]+)",
    re.IGNORECASE
)

# 2. Vocabulário Clínico Estruturado (Alinhado ao Mermaid em steps.md)
ENTITY_CATEGORIES = {
    "Symptom": {
        "lower quadrant abdominal pain", "abdominal pain", "nausea", "constipation",
        "pain", "dyspnea", "fever", "chest pain", "shortness of breath", "headache"
    },
    "Disease": {
        "mucinous pancreatic cystic neoplasm", "mucinous pancreatic cyst",
        "gastric duplication cyst", "GDC [gastric duplication cyst]",
        "systemic lupus erythematosus", "ventricular septal defect",
        "malignancy", "adenocarcinoma", "heart failure", "tuberculosis"
    },
    "Exam": {
        "contrast enhanced computed tomography", "computed tomography",
        "EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]",
        "FNA [fine needle aspiration]", "Intraoperative endoscopy",
        "oesophagogastroduodenoscopy", "Final pathology", "physical examination",
        "biopsy", "ultrasound"
    },
    "Result": {
        "cystic lesion", "pancreatic echotexture", "cyst",
        "extracellular mucin", "carcinoembryonic antigen",
        "carbohydrate antigen", "internal septations", "associated masses"
    },
    "Anatomy": {
        "right flank", "stomach", "pancreas", "posterior wall",
        "lesser sac", "coeliac axis", "coeliac vessels", "abdomen"
    },
    "Medication": {
        "antibiotics", "chemotherapy", "isoniazid", "rifampin",
        "ethambutol", "pyrazinamide", "aspirin", "omeprazole"
    }
}

TERM_TO_TYPE = {
    term.lower(): cat
    for cat, terms in ENTITY_CATEGORIES.items()
    for term in terms
}

def is_entity_negated(entity_str: str, text: str) -> bool:
    """Verifica se a entidade está sob escopo de negação ou resolução clínica."""
    clauses = re.split(r"\s+(?:but|however|although)\s+", text, flags=re.IGNORECASE)
    for clause in clauses:
        m = NEGATION_PATTERN.search(clause)
        if m and entity_str.lower() in m.group(1).lower():
            return True
    return False

def extract_knowledge_graph_rich(case_row, sentence_list):
    """
    Constrói os DataFrames de nós e arestas com distribuição semântica rica de atributos:
    - Atributos intrínsecos nos NÓS.
    - Atributos relacionais nas ARESTAS.
    """
    nodes = []
    edges = []
    node_ids = {}

    def get_or_create_node(label, node_type, attributes=""):
        key = (label.lower(), node_type)
        if key in node_ids:
            for node in nodes:
                if node["node_id"] == node_ids[key]:
                    if not node["attributes"] and attributes:
                        node["attributes"] = attributes
                    elif attributes and attributes not in node["attributes"]:
                        node["attributes"] += f"; {attributes}"
            return node_ids[key]
        prefix = {
            "Patient": "P",
            "Symptom": "S",
            "Disease": "D",
            "Exam": "E",
            "Result": "R",
            "Anatomy": "A",
            "Medication": "M",
            "Effect": "Ef"
        }.get(node_type, "N")
        nid = f"{prefix}{len([k for k in node_ids if k[1] == node_type]) + 1}"
        node_ids[key] = nid
        nodes.append({
            "node_id": nid,
            "type": node_type,
            "label": label,
            "attributes": attributes
        })
        return nid

    def add_edge(source_id, target_id, relation, attributes=""):
        eid = f"e{len(edges) + 1}"
        edges.append({
            "edge_id": eid,
            "source_id": source_id,
            "target_id": target_id,
            "relation": relation,
            "attributes": attributes
        })
        return eid

    # 1. Nó do Paciente: atributos demográficos intrínsecos
    patient_attrs = f"age={case_row['age']}; gender={case_row['gender']}"
    patient_id = get_or_create_node(
        f"Patient {case_row['case_id']}",
        "Patient",
        patient_attrs
    )

    # 2. Processamento sentencial com contextualização de atributos
    for s in sentence_list:
        txt = s.text.lower()
        orig_text = s.text

        if "non-contributory" in txt or "were normal" in txt:
            continue

        found_entities = []
        for tok in s.tokens:
            tok_lower = tok.lower()
            if tok_lower in TERM_TO_TYPE:
                found_entities.append((tok, TERM_TO_TYPE[tok_lower]))

        active_entities = [e for e in found_entities if not is_entity_negated(e[0], orig_text)]

        symptoms = [e[0] for e in active_entities if e[1] == "Symptom"]
        diseases = [e[0] for e in active_entities if e[1] == "Disease"]
        exams = [e[0] for e in active_entities if e[1] == "Exam"]
        results = [e[0] for e in active_entities if e[1] == "Result"]
        anatomy = [e[0] for e in active_entities if e[1] == "Anatomy"]
        meds = [e[0] for e in active_entities if e[1] == "Medication"]

        # Extração de Dimensões
        sizes = []
        spans_sz = set()
        for m in DIMENSION_3D.finditer(orig_text):
            sizes.append((m.start(), m.end(), m.group(0).strip()))
            spans_sz.update(range(m.start(), m.end()))
        for m in DIMENSION_2D.finditer(orig_text):
            if not any(i in spans_sz for i in range(m.start(), m.end())):
                sizes.append((m.start(), m.end(), m.group(0).strip()))
                spans_sz.update(range(m.start(), m.end()))
        for m in DIMENSION_1D.finditer(orig_text):
            if not any(i in spans_sz for i in range(m.start(), m.end())):
                sizes.append((m.start(), m.end(), m.group(0).strip()))

        res_spans = []
        for r in results:
            m = re.search(re.escape(r), orig_text, re.IGNORECASE)
            if m:
                res_spans.append((m.start(), m.end(), r))

        assigned_sizes = {}
        for s_start, s_end, sz in sizes:
            if res_spans:
                def dist_sz(ent):
                    start, end, _ = ent
                    if end <= s_start: return s_start - end
                    if start >= s_end: return start - s_end
                    return 0
                closest = min(res_spans, key=dist_sz)
                if dist_sz(closest) < 30:
                    assigned_sizes[closest[2]] = sz

        labs = [(m.start(), m.end(), m.group(1), m.group(2)) for m in LAB_MEASUREMENT.finditer(orig_text)]
        assigned_labs = {r: [] for r in results}
        for l_start, l_end, val, unit in labs:
            preceding = [ent for ent in res_spans if ent[1] <= l_start]
            if preceding:
                closest = min(preceding, key=lambda e: l_start - e[1])
                if (l_start - closest[1]) < 60:
                    assigned_labs[closest[2]].append(f"value={val}; unit={unit}")

        dosages = [(m.start(), m.end(), m.group(0).strip()) for m in DOSAGE.finditer(orig_text)]

        dur_match = DURATION_HISTORY.search(orig_text) or DURATION_GENERIC.search(orig_text)
        duration_val = dur_match.group(0) if dur_match else ""

        pod_match = TIMELINE_POD.search(orig_text)
        timeline_val = pod_match.group(0) if pod_match else ""

        lat_match = LATERALITY.search(orig_text)
        laterality_val = lat_match.group(0).lower() if lat_match else ""

        asp_match = ASPECT.search(orig_text)
        aspect_val = asp_match.group(0).lower() if asp_match else ""

        # 1. Relação: Patient --PRESENTS_WITH--> Symptom
        if symptoms and any(w in txt for w in ["presented", "history", "pain", "complained", "had", "symptom"]):
            pres_attr = f"duration={duration_val}" if duration_val else ""
            for sym in symptoms:
                sid = get_or_create_node(sym, "Symptom")
                add_edge(patient_id, sid, "PRESENTS_WITH", pres_attr)
                if "pain" in sym.lower():
                    for anat in anatomy:
                        loc_attr = f"laterality={laterality_val}" if laterality_val else ""
                        aid = get_or_create_node(anat, "Anatomy")
                        add_edge(sid, aid, "LOCATED_IN", loc_attr)

        # 2. Relação: Patient --TAKES--> Medication
        if meds:
            for med in meds:
                mid = get_or_create_node(med, "Medication")
                m_match = re.search(re.escape(med), orig_text, re.IGNORECASE)
                dose_attr = ""
                if m_match and dosages:
                    closest_dose = sorted(dosages, key=lambda d: abs(d[0] - m_match.end()))[0][2]
                    dose_attr = f"dosage={closest_dose}"
                elif dosages:
                    dose_attr = f"dosage={dosages[0][2]}"
                add_edge(patient_id, mid, "TAKES", dose_attr)

        # 3. Relação: Patient --UNDERWENT_EXAM--> Exam
        if exams:
            for ex in exams:
                ex_edge_attr = f"timeline={timeline_val}" if timeline_val else ""
                eid = get_or_create_node(ex, "Exam")
                add_edge(patient_id, eid, "UNDERWENT_EXAM", ex_edge_attr)

                for res in results:
                    res_attrs = []
                    if res in assigned_sizes:
                        res_attrs.append(f"size={assigned_sizes[res]}")
                    if assigned_labs.get(res):
                        res_attrs.extend(assigned_labs[res])
                    if "normal" in txt and "echotexture" in res.lower():
                        res_attrs.append("status=normal")

                    rid = get_or_create_node(res, "Result", "; ".join(res_attrs))
                    add_edge(eid, rid, "REVEALS")

                    for anat in anatomy:
                        loc_attr = f"aspect={aspect_val}" if aspect_val else ""
                        aid = get_or_create_node(anat, "Anatomy")
                        add_edge(rid, aid, "LOCATED_IN", loc_attr)

        # 4. Relação: Patient --DIAGNOSED_WITH / HAS_HISTORY--> Disease
        if diseases:
            for dis in diseases:
                did = get_or_create_node(dis, "Disease")
                if "history" in txt and ("medical" in txt or "past" in txt):
                    rel = "HAS_FAMILY_HISTORY" if "family" in txt else "HAS_PERSONAL_HISTORY"
                    edge_attr = ""
                else:
                    rel = "DIAGNOSED_WITH"
                    diag_attrs = []
                    if "pathology" in txt or "final pathology" in txt:
                        diag_attrs.append("status=confirmed")
                        diag_attrs.append("method=pathology")
                    elif "suggesting" in txt or "suspected" in txt:
                        diag_attrs.append("status=suspected")
                        diag_attrs.append("method=clinical/imaging")
                    else:
                        diag_attrs.append("status=confirmed")
                    edge_attr = "; ".join(diag_attrs)

                add_edge(patient_id, did, rel, edge_attr)

                for anat in anatomy:
                    loc_attr = f"aspect={aspect_val}" if aspect_val else ""
                    aid = get_or_create_node(anat, "Anatomy")
                    add_edge(did, aid, "LOCATED_IN", loc_attr)

    return pd.DataFrame(nodes), pd.DataFrame(edges)

nodes_df, edges_df = extract_knowledge_graph_rich(case, sentences)

proc_dir = cfg.PROCESSED_DATA_DIR
nodes_df.to_csv(proc_dir / "nodesCase2.csv", index=False)
edges_df.to_csv(proc_dir / "edgesCase2.csv", index=False)

print("Grafo do Caso 2 construído!")
print(f"Total de Nós: {len(nodes_df)}")
print(f"Total de Arestas: {len(edges_df)}")


Grafo do Caso 2 construído!
Total de Nós: 7
Total de Arestas: 6


In [5]:
print("=== TABELA DE NÓS (nodesCase2.csv) ===")
display(nodes_df)

print("\n=== TABELA DE ARESTAS COM ATRIBUTOS (edgesCase2.csv) ===")
display(edges_df)


=== TABELA DE NÓS (nodesCase2.csv) ===


,node_id,type,label,attributes
0,P1,Patient,Patient PMC3437073_01,age=55.0; gender=Male
1,D1,Disease,tuberculosis,
2,E1,Exam,Ultrasound,
3,M1,Medication,Isoniazid,
4,M2,Medication,Rifampin,
5,M3,Medication,Ethambutol,
6,M4,Medication,Pyrazinamide,



=== TABELA DE ARESTAS COM ATRIBUTOS (edgesCase2.csv) ===


,edge_id,source_id,target_id,relation,attributes
0,e1,P1,D1,DIAGNOSED_WITH,status=confirmed
1,e2,P1,E1,UNDERWENT_EXAM,
2,e3,P1,M1,TAKES,dosage=300 mg
3,e4,P1,M2,TAKES,dosage=600 mg
4,e5,P1,M3,TAKES,dosage=600 mg
5,e6,P1,M4,TAKES,dosage=500 mg


In [6]:
from IPython.display import display, Markdown

def generate_mermaid_code(nodes, edges):
    lines = ["```mermaid", "graph LR"]
    for _, r in nodes.iterrows():
        nid = r["node_id"]
        lbl = str(r["label"]).replace('"', "'")
        ntype = r["type"]
        attrs = str(r["attributes"]).replace('"', "'") if pd.notna(r.get("attributes")) and r.get("attributes") else ""
        attr_str = f"<br/><small>{attrs}</small>" if attrs else ""
        
        if ntype == "Patient":
            lines.append(f'    {nid}(("{lbl}{attr_str}"))')
        else:
            lines.append(f'    {nid}["{lbl} <br/> <i>({ntype})</i>{attr_str}"]')

    for _, r in edges.iterrows():
        src = r["source_id"]
        tgt = r["target_id"]
        rel = r["relation"]
        attrs = str(r["attributes"]).replace('"', "'") if pd.notna(r.get("attributes")) and r.get("attributes") else ""
        rel_label = f"{rel}<br/><i>[{attrs}]</i>" if attrs else rel
        lines.append(f'    {src} -- "{rel_label}" --> {tgt}')
    lines.append("```")
    return "\n".join(lines)

mermaid_md = generate_mermaid_code(nodes_df, edges_df)
display(Markdown(mermaid_md))
print(mermaid_md)


```mermaid
graph LR
    P1(("Patient PMC3437073_01<br/><small>age=55.0; gender=Male</small>"))
    D1["tuberculosis <br/> <i>(Disease)</i>"]
    E1["Ultrasound <br/> <i>(Exam)</i>"]
    M1["Isoniazid <br/> <i>(Medication)</i>"]
    M2["Rifampin <br/> <i>(Medication)</i>"]
    M3["Ethambutol <br/> <i>(Medication)</i>"]
    M4["Pyrazinamide <br/> <i>(Medication)</i>"]
    P1 -- "DIAGNOSED_WITH<br/><i>[status=confirmed]</i>" --> D1
    P1 -- "UNDERWENT_EXAM" --> E1
    P1 -- "TAKES<br/><i>[dosage=300 mg]</i>" --> M1
    P1 -- "TAKES<br/><i>[dosage=600 mg]</i>" --> M2
    P1 -- "TAKES<br/><i>[dosage=600 mg]</i>" --> M3
    P1 -- "TAKES<br/><i>[dosage=500 mg]</i>" --> M4
```

```mermaid
graph LR
    P1(("Patient PMC3437073_01<br/><small>age=55.0; gender=Male</small>"))
    D1["tuberculosis <br/> <i>(Disease)</i>"]
    E1["Ultrasound <br/> <i>(Exam)</i>"]
    M1["Isoniazid <br/> <i>(Medication)</i>"]
    M2["Rifampin <br/> <i>(Medication)</i>"]
    M3["Ethambutol <br/> <i>(Medication)</i>"]
    M4["Pyrazinamide <br/> <i>(Medication)</i>"]
    P1 -- "DIAGNOSED_WITH<br/><i>[status=confirmed]</i>" --> D1
    P1 -- "UNDERWENT_EXAM" --> E1
    P1 -- "TAKES<br/><i>[dosage=300 mg]</i>" --> M1
    P1 -- "TAKES<br/><i>[dosage=600 mg]</i>" --> M2
    P1 -- "TAKES<br/><i>[dosage=600 mg]</i>" --> M3
    P1 -- "TAKES<br/><i>[dosage=500 mg]</i>" --> M4
```


## 🚨 Análise Crítica dos Gargalos e Limitações Observados no Caso 2

O Caso 2 (`PMC3437073_01`) é um relato paradigmático que revela as **fronteiras e fragilidades** da modelagem de grafos clínicos clássicos baseados em regras e ontologias fechadas. Ao inspecionarmos os nós, as arestas e o texto clínico de origem, identificamos **6 grandes gargalos**:

---

### 1. Falso Positivo por Negação de Contato Epidemiológico
* **Texto Original (Sentença 1):** *"He had no other medical problems and no history of contact with a tuberculous patient."*
* **Comportamento do Grafo:** Gerou imediatamente `P1 -- DIAGNOSED_WITH [status=confirmed] --> D1 (tuberculosis)`.
* **Gargalo Clínico:** A palavra *"tuberculous"* qualificava o substantivo *"patient"* em uma oração de negação de exposição epidemiológica (*"no history of contact with..."*). A modelagem baseada em regras de casamento de termos interpretou a simples menção ao termo como se o próprio paciente tivesse a doença confirmada, criando uma alucinação clínica grave na primeira sentença.

---

### 2. Colapso da Linha Temporal Farmacoterapêutica (Fase Intensiva vs. Fase de Manutenção)
* **Texto Original (Sentença 16):** *"The patient was started on Isoniazid (INH) 300 mg OD, Rifampin 600 mg OD, Ethambutol 600 mg OD and Pyrazinamide 500 mg OD for 5 weeks, then maintained on the former two drugs."*
* **Comportamento do Grafo:** Conectou estaticamente o paciente aos 4 fármacos com arestas idênticas `TAKES`:
  * `P1 -- TAKES [dosage=300 mg] --> Isoniazid`
  * `P1 -- TAKES [dosage=600 mg] --> Rifampin`
  * `P1 -- TAKES [dosage=600 mg] --> Ethambutol`
  * `P1 -- TAKES [dosage=500 mg] --> Pyrazinamide`
* **Gargalo Estrutural:** Grafos de conhecimento estáticos **não possuem dimensão temporal intrínseca**. O modelo não consegue expressar que:
  * Etambutol e Pirazinamida foram **interrompidos após 5 semanas** (fase de ataque do tratamento de TB);
  * Isoniazida e Rifampicina foram mantidas na fase de consolidação.
  Para um sistema de apoio à decisão, o paciente aparenta estar em polifarmácia quadrupla perpétua.

---

### 3. Confusão entre Infecção Passada Inativa vs. Complicação Aguda Presente
* **Texto Original (Sentenças 5 e 9):**
  * Sentença 5: *"Chest X-rays showed a calcified lesion near the hilum of the right lung suggestive of an **old pulmonary tuberculosis**."*
  * Sentença 9: *"...a complicated **tuberculous splenic abscess** was the most probable diagnosis."*
* **Gargalo Semântico:** O paciente possui uma cicatriz inativa calcificada pulmonar de infecção antiga (*"old pulmonary tuberculosis"*), mas a patologia aguda responsável pela internação é um abscesso esplênico tuberculoso. Como ambos contêm termos relacionados a tuberculose, o grafo unifica tudo sob um único nó `tuberculosis`, perdendo a distinção crítica entre doença pregressa cicatrizada e complicação extrapulmonar ativa.

---

### 4. Incapacidade de Representar Relações Topológicas Inter-Lesão (Fístula)
* **Texto Original (Sentença 8 e 12):**
  * *"CT scan of the abdomen revealed a splenic abscess communicating with another subcutaneous abscess through the lower chest wall."*
  * *"There was an abscess cavity within the spleen communicating with the subcutaneous abscess through a small tract in the lower chest wall..."*
* **Gargalo Ontológico:** No esquema atual, cada lesão se conecta independentemente à sua anatomia (`Result -- LOCATED_IN --> Anatomy`). No entanto, o achado clínico fundamental é a **comunicação física (trajeto fistuloso transdiafragmático)** entre o abscesso do baço e o abscesso subcutâneo da parede torácica. O modelo não possui primitives para relações do tipo `Result -- COMMUNICATES_WITH --> Result` ou `FISTULA_TRACT`.

---

### 5. Coexistência de Resultados Negativos com Confirmação Histológica
* **Texto Original (Sentenças 14 e 15):**
  * Sentença 14: *"Histopathology of the spleen revealed the characteristic tuberculoid granuloma with epithelioid cells, Langhans' multinucleated giant cells, and caseation necrosis..."*
  * Sentença 15: *"Acid fast bacilli staining, tuberculous and bacterial cultures were all negative."*
* **Gargalo Diagnóstico:** As culturas e colorações bacterianas foram todas negativas (ausência de bacilos viáveis), mas o diagnóstico foi fechado pela histopatologia clássica (necrose de caseificação). A modelagem binária tem enorme dificuldade em representar que um exame foi realizado e resultou negativo sem que isso anule o diagnóstico clínico.

---

### 6. Rigidez Lexical e Perda da Queixa Principal
* **Texto Original (Sentença 0):** *"A 55-year-old male presented with a gradually increasing mass on the left lateral lower chest and upper abdomen over a period of one year."*
* **Gargalo de Cobertura:** A queixa primária do paciente é uma **massa** há 1 ano. Como "mass" foi classificada apenas no contexto de achados cirúrgicos/oncológicos e não no vocabulário estático de sintomas de apresentação, o grafo resultante possui **ZERO sintomas** associados a `P1`, gerando uma representação clínica empobrecida.
